# Train CafeBERT trên ViGoEmotions 7 nhóm Ekman




## 1. Kiểm tra GPU

In [ ]:
import sys, subprocess, importlib.util
print('Python:', sys.version.split()[0], '|', sys.executable)
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True, timeout=30).stdout or 'nvidia-smi: không có output')
except FileNotFoundError:
    print('nvidia-smi: KHÔNG TÌM THẤY (máy không có GPU NVIDIA hoặc chưa cài driver)')
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# Thư viện cần thêm (kernel vigoenv đã có PyTorch CUDA). Bỏ qua nếu đã cài.
%pip install -q transformers==4.52.4 sentencepiece protobuf openpyxl scikit-learn pandas matplotlib

## 2. Cấu hình

In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# ======================= CHỈNH Ở ĐÂY =======================
MODEL_KEY = 'cafebert'
DATA_TAG = 'data_ekman7'
VISOBERT_RUN_NAME = 'visobert_ekman7_seed42_pooler'
# ===========================================================

# Cấu trúc thống nhất:
# repo/
#   data/processed/<DATA_TAG>/
#   notebooks/
#   runs/<RUN_NAME>/

COLAB_REPO_ROOT = '/content/drive/MyDrive/DAP391m-sentiment-vn'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path(COLAB_REPO_ROOT)
else:
    CWD = Path.cwd().resolve()
    if (CWD / 'notebooks').is_dir():
        REPO_ROOT = CWD
    elif CWD.name == 'notebooks' and (CWD.parent / 'notebooks').is_dir():
        REPO_ROOT = CWD.parent
    else:
        raise RuntimeError(
            f'Không xác định được repo root từ {CWD}. '
            'Hãy chạy notebook từ thư mục notebooks/ hoặc repo root.'
        )

if not (REPO_ROOT / 'notebooks').is_dir():
    raise RuntimeError(f'Không tìm thấy thư mục notebooks trong repo: {REPO_ROOT}')

DATA_ROOT = (REPO_ROOT / 'data' / 'processed').as_posix()
OUT_ROOT = (REPO_ROOT / 'runs').as_posix()
DATA_DIR = f'{DATA_ROOT}/{DATA_TAG}'

# Cho phép ghi đè khi chạy thử tự động; bình thường không cần đặt các biến này.
DATA_DIR = os.environ.get('VIGO_DATA_DIR', DATA_DIR)
OUT_ROOT = os.environ.get('VIGO_OUT_ROOT', OUT_ROOT)
VISOBERT_RUN = os.path.join(OUT_ROOT, VISOBERT_RUN_NAME)

PRESET = dict(model_name='uitnlp/CafeBERT', lr=2e-5, batch_size=16, grad_accum=2, use_amp=True, grad_ckpt=True)
SEED, EPOCHS, MAX_LEN, THRESHOLD, POOLING = 42, 12, 200, 0.5, 'pooler'
USE_PYVI = False
RESUME = True
DELETE_LAST_WHEN_DONE = True
SMOKE_TEST = False

SMOKE_TEST = os.environ.get('VIGO_SMOKE', '1' if SMOKE_TEST else '0') == '1'

P = PRESET
MODEL_NAME, LR, BATCH_SIZE, GRAD_ACCUM, USE_AMP, GRAD_CKPT = (
    P[k] for k in ('model_name', 'lr', 'batch_size', 'grad_accum', 'use_amp', 'grad_ckpt')
)
MODEL_NAME = os.environ.get('VIGO_MODEL_NAME', MODEL_NAME)
if SMOKE_TEST:
    EPOCHS = 1

# GPU nhỏ: giảm bộ nhớ sử dụng nếu cần.
import torch, importlib.util
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
FREEZE_WORD_EMB, OPTIM = False, 'adamw'
LOWMEM = os.environ.get('VIGO_LOWMEM', '1') == '1'
if LOWMEM and 0 < VRAM_GB < 10:
    BATCH_SIZE, GRAD_ACCUM, USE_AMP, GRAD_CKPT, FREEZE_WORD_EMB = 8, 4, True, True, True
    OPTIM = 'adamw8bit' if importlib.util.find_spec('bitsandbytes') else 'adafactor'

RUN_NAME = f"{MODEL_TAG}_s{SEED}_{POOLING}"
OUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

for f in ['train.csv', 'val.csv', 'test.csv', 'label_dict_ekman7.json', 'ekman_mapping.json']:
    assert os.path.exists(os.path.join(DATA_DIR, f)), (
        f'Thiếu {os.path.join(DATA_DIR, f)} — kiểm tra DATA_TAG hoặc chạy notebook tiền xử lý.'
    )

CONFIG = dict(seed=SEED, epochs=EPOCHS, batch_size=BATCH_SIZE * GRAD_ACCUM, micro_batch=BATCH_SIZE, grad_accum=GRAD_ACCUM,
              lr=LR, max_len=MAX_LEN, threshold=THRESHOLD, model_name=MODEL_NAME, use_pyvi=USE_PYVI, pooling=POOLING,
              use_amp=USE_AMP, grad_ckpt=GRAD_CKPT, dynamic_padding=True, loss='bce_pos_weight', smoke=SMOKE_TEST,
              optimizer=OPTIM, freeze_word_embeddings=FREEZE_WORD_EMB,
              label_scheme='ekman7', data_tag=DATA_TAG)
with open(os.path.join(OUT_DIR, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=2)
print('Repo root:', REPO_ROOT)
print('Run:', RUN_NAME)
print('Dữ liệu:', DATA_DIR)
print('Output dir:', OUT_DIR)
print('ViSoBERT run:', VISOBERT_RUN)
print(f'VRAM: {VRAM_GB:.1f} GB')
print(json.dumps(CONFIG, indent=1))


## 3. Import, seed, ghi lại môi trường

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
import transformers
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda' and not SMOKE_TEST:
    raise SystemExit('Không thấy GPU. Kiểm tra kernel vigoenv và driver NVIDIA.')
USE_AMP = USE_AMP and device.type == 'cuda'
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

env = dict(python=sys.version.split()[0], torch=torch.__version__, transformers=transformers.__version__,
           pandas=pd.__version__, numpy=np.__version__,
           gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else None)
import sklearn; env['scikit-learn'] = sklearn.__version__
with open(os.path.join(OUT_DIR, 'env.json'), 'w', encoding='utf-8') as f:
    json.dump(env, f, indent=2)
print(env)

## 4. Dữ liệu `data_ekman7`

In [ ]:
label_dict = json.load(open(os.path.join(DATA_DIR, 'label_dict_ekman7.json'), encoding='utf-8'))
LABEL_NAMES = [label_dict[str(i)] for i in range(len(label_dict))]
N_LABELS = len(LABEL_NAMES)
TEXT_COL = 'model_text' if USE_PYVI else 'text'

def load(split):
    d = pd.read_csv(os.path.join(DATA_DIR, f'{split}.csv'), keep_default_na=False, dtype={'id': str})
    Y = np.zeros((len(d), N_LABELS), dtype=np.float32)
    for i, l in enumerate(d['labels']):
        Y[i, json.loads(l)] = 1.0
    return d, Y

(train_df, Y_train), (val_df, Y_val), (test_df, Y_test) = load('train'), load('val'), load('test')
assert (len(train_df), len(val_df), len(test_df)) == (16531, 2066, 2067), 'Số câu khác data_ekman7 chuẩn'
if SMOKE_TEST:
    train_df, Y_train = train_df.head(48), Y_train[:48]
    val_df, Y_val = val_df.head(24), Y_val[:24]
    test_df, Y_test = test_df.head(24), Y_test[:24]
print('Nhãn:', LABEL_NAMES)
print('Kích thước:', len(train_df), len(val_df), len(test_df), '| cột văn bản:', TEXT_COL)
print('Số câu dương mỗi nhóm (train):', dict(zip(LABEL_NAMES, Y_train.sum(0).astype(int))))
print('Ví dụ:', train_df[TEXT_COL].iloc[0])

## 5. Tokenize (một lần) và DataLoader với padding động

In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception as e:
    print('Fast tokenizer lỗi, dùng bản chậm:', e)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def encode(texts):
    return tokenizer(list(texts), truncation=True, max_length=MAX_LEN, padding=False,
                     return_attention_mask=False, return_token_type_ids=False)['input_ids']

t0 = time.time()
tr_ids, va_ids, te_ids = encode(train_df[TEXT_COL]), encode(val_df[TEXT_COL]), encode(test_df[TEXT_COL])
lens = np.array([len(x) for x in tr_ids])
print(f'Tokenize xong ({time.time()-t0:.1f}s). Token/câu (train): trung bình {lens.mean():.1f} | p99 {np.percentile(lens, 99):.0f} | '
      f'bị cắt ở {MAX_LEN}: {(lens >= MAX_LEN).mean()*100:.2f}%')

class DS(Dataset):
    def __init__(self, ids, Y): self.ids, self.Y = ids, Y
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.Y[i]

PAD = tokenizer.pad_token_id
def collate(batch):
    L = max(len(x) for x, _ in batch)
    ids = torch.full((len(batch), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for r, (x, _) in enumerate(batch):
        ids[r, :len(x)] = torch.tensor(x); mask[r, :len(x)] = 1
    return ids, mask, torch.from_numpy(np.stack([y for _, y in batch]))

train_gen = torch.Generator(); train_gen.manual_seed(SEED)
train_loader = DataLoader(DS(tr_ids, Y_train), batch_size=BATCH_SIZE, shuffle=True, generator=train_gen, collate_fn=collate, num_workers=0)
val_loader = DataLoader(DS(va_ids, Y_val), batch_size=64, shuffle=False, collate_fn=collate, num_workers=0)
test_loader = DataLoader(DS(te_ids, Y_test), batch_size=64, shuffle=False, collate_fn=collate, num_workers=0)
print('Batch/epoch:', len(train_loader), '| bước cập nhật/epoch:', -(-len(train_loader) // GRAD_ACCUM))

## 6. Mô hình (giống run ViSoBERT: encoder + dropout 0,2 + Linear, dùng `pooler_output`)

In [ ]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_name, pooling='pooler'):
        super().__init__()
        self.pooling = pooling
        config = AutoConfig.from_pretrained(model_name, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1)
        self.bert = AutoModel.from_pretrained(model_name, config=config)   # cảnh báo "pooler newly initialized" là bình thường
        self.drop = nn.Dropout(p=0.2)
        self.fc = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        x = out.pooler_output if self.pooling == 'pooler' else out.last_hidden_state[:, 0]
        return self.fc(self.drop(x))

set_seed(SEED)
model = ModelSentimentClassifier(N_LABELS, MODEL_NAME, POOLING).to(device)
if GRAD_CKPT:
    model.bert.gradient_checkpointing_enable()
if FREEZE_WORD_EMB:
    model.bert.embeddings.word_embeddings.weight.requires_grad_(False)
print(f'Tham số: {sum(p.numel() for p in model.parameters()):,} (train {sum(p.numel() for p in model.parameters() if p.requires_grad):,}) '
      f'| AMP {USE_AMP} | grad checkpointing {GRAD_CKPT} | optimizer {OPTIM}')

## 7. Optimizer, scheduler, loss

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
if OPTIM == 'adamw8bit':
    import bitsandbytes as bnb
    optimizer = bnb.optim.AdamW8bit(params, lr=LR)
elif OPTIM == 'adafactor':
    from transformers.optimization import Adafactor
    optimizer = Adafactor(params, lr=LR, scale_parameter=False, relative_step=False, warmup_init=False)
else:
    optimizer = AdamW(params, lr=LR)
steps_per_epoch = -(-len(train_loader) // GRAD_ACCUM)
scheduler = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=steps_per_epoch,
                          num_training_steps=steps_per_epoch * EPOCHS)
cnt = np.clip(Y_train.sum(0), 1, None)
pos_weight = torch.tensor((len(Y_train) - cnt) / cnt, dtype=torch.float32).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
print('pos_weight:', dict(zip(LABEL_NAMES, pos_weight.cpu().numpy().round(2))))

## 8. Hàm train / eval

In [ ]:
def run_epoch(loader, train=False, return_probs=False):
    model.train(train)
    losses, all_t, all_p = [], [], []
    t_ep = time.time()
    if train:
        optimizer.zero_grad(set_to_none=True)
    with (torch.enable_grad() if train else torch.no_grad()):
        for step, (ids, mask, y) in enumerate(loader):
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(ids, mask)
            loss = loss_fn(logits.float(), y)
            if train:
                scaler.scale(loss / GRAD_ACCUM).backward()
                if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(loader):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer); scaler.update(); scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                if (step + 1) % 100 == 0:
                    print(f'  bước {step+1}/{len(loader)} | loss {np.mean(losses[-100:]):.4f} | {time.time()-t_ep:.0f}s', flush=True)
            losses.append(loss.item())
            all_t.append(y.cpu().numpy()); all_p.append(torch.sigmoid(logits.float()).detach().cpu().numpy())
    T, Pr = np.vstack(all_t), np.vstack(all_p)
    pred = (Pr >= THRESHOLD).astype(int)
    out = (float(np.mean(losses)), float((T == pred).mean()), float(f1_score(T, pred, average='macro', zero_division=0)))
    return out + (Pr, T) if return_probs else out

def save_atomic(obj, path):
    tmp = path + '.tmp'; torch.save(obj, tmp); os.replace(tmp, path)

## 9. Huấn luyện (resume được)

Bị ngắt thì `Run All` lại: notebook đọc `last.pt` và chạy tiếp. Nếu sau 2 epoch Macro-F1 val vẫn dưới 0,15, notebook dừng và báo nên giảm `lr` (dấu hiệu mô hình không hội tụ).

In [ ]:
LAST, BEST, HIST, DONE = (os.path.join(OUT_DIR, n) for n in ['last.pt', 'best.pt', 'history.json', 'DONE'])
history, best_f1, start_epoch = [], 0.0, 0

if os.path.exists(DONE) and RESUME:
    print('Run này đã xong trước đó, bỏ qua huấn luyện (xóa file DONE để chạy lại).')
    history = json.load(open(HIST)); best_f1 = max(h['val_f1'] for h in history); start_epoch = EPOCHS
elif os.path.exists(LAST) and RESUME:
    ck = torch.load(LAST, map_location='cpu', weights_only=False)
    _extra = {k: v for k, v in CONFIG.items() if k not in ck['config']}   # khóa mới thêm sau: chỉ chấp nhận giá trị mặc định
    assert {k: CONFIG.get(k) for k in ck['config']} == ck['config'] and all(v in (False, 'adamw') for v in _extra.values()), \
        f"Cấu hình checkpoint khác cấu hình hiện tại:\n{ck['config']}\n{CONFIG}"
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
    history, best_f1, start_epoch = ck['history'], ck['best_f1'], ck['epoch']
    train_gen.set_state(ck['gen_state']); torch.set_rng_state(ck['rng_cpu'])
    if torch.cuda.is_available() and ck['rng_cuda'] is not None: torch.cuda.set_rng_state_all(ck['rng_cuda'])
    random.setstate(ck['rng_py']); np.random.set_state(ck['rng_np'])
    del ck
    print(f'Resume từ epoch {start_epoch + 1}/{EPOCHS} (best val Macro-F1: {best_f1:.4f})')

t_start = time.time()
for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader)
    history.append(dict(epoch=epoch + 1, lr=optimizer.param_groups[0]['lr'], train_loss=tr_loss, train_acc=tr_acc, train_f1=tr_f1,
                        val_loss=va_loss, val_acc=va_acc, val_f1=va_f1))
    if va_f1 > best_f1:
        best_f1 = va_f1
        save_atomic(dict(model=model.state_dict(), epoch=epoch + 1, val_f1=va_f1, config=CONFIG), BEST)
    save_atomic(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                     scaler=scaler.state_dict(), epoch=epoch + 1, history=history, best_f1=best_f1, config=CONFIG,
                     gen_state=train_gen.get_state(), rng_cpu=torch.get_rng_state(),
                     rng_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                     rng_py=random.getstate(), rng_np=np.random.get_state()), LAST)
    json.dump(history, open(HIST, 'w'), indent=1)
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | train loss {tr_loss:.4f} F1 {tr_f1:.4f} | val loss {va_loss:.4f} F1 {va_f1:.4f}'
          f'{"  *best*" if va_f1 == best_f1 else ""} | {(time.time()-t0)/60:.1f} phút')
    if epoch + 1 >= 2 and best_f1 < 0.15 and not SMOKE_TEST:
        raise SystemExit(f'Macro-F1 val chỉ {best_f1:.3f} sau {epoch+1} epoch: mô hình không hội tụ. '
                         f'Giảm lr (vd. 1e-5) trong PRESET (cell cấu hình), xóa thư mục {OUT_DIR} rồi chạy lại.')

if start_epoch < EPOCHS:
    open(DONE, 'w').write('ok')
    if DELETE_LAST_WHEN_DONE and os.path.exists(LAST):
        os.remove(LAST)
    print(f'Xong huấn luyện sau {(time.time()-t_start)/60:.1f} phút. Best val Macro-F1 = {best_f1:.4f}')

## 10. Đường cong huấn luyện

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['epoch'], h['train_loss'], label='train'); ax[0].plot(h['epoch'], h['val_loss'], label='val'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid()
ax[1].plot(h['epoch'], h['train_f1'], label='train'); ax[1].plot(h['epoch'], h['val_f1'], label='val'); ax[1].set_title('Macro-F1 @ 0.5'); ax[1].legend(); ax[1].grid()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'learning_curve.png'), dpi=120); plt.show()

## 11. Đánh giá trên test, lưu xác suất và kết quả

In [ ]:
ck = torch.load(BEST, map_location='cpu', weights_only=False)
model.load_state_dict(ck['model']); model.to(device)
print(f"Checkpoint tốt nhất: epoch {ck['epoch']} (val Macro-F1 {ck['val_f1']:.4f})"); del ck

_, _, _, P_val, T_val = run_epoch(val_loader, return_probs=True)
_, _, _, P_test, T_test = run_epoch(test_loader, return_probs=True)
np.save(os.path.join(OUT_DIR, 'val_probs.npy'), P_val); np.save(os.path.join(OUT_DIR, 'val_labels.npy'), T_val)
np.save(os.path.join(OUT_DIR, 'test_probs.npy'), P_test); np.save(os.path.join(OUT_DIR, 'test_labels.npy'), T_test)

rep = classification_report(T_test, (P_test >= THRESHOLD).astype(int), target_names=LABEL_NAMES, zero_division=0, output_dict=True)
pd.DataFrame(rep).transpose().to_excel(os.path.join(OUT_DIR, 'classification_report_test.xlsx'))
print(classification_report(T_test, (P_test >= THRESHOLD).astype(int), target_names=LABEL_NAMES, zero_division=0))

def scores(T, Pr):
    p = (Pr >= THRESHOLD).astype(int)
    return {k: round(f1_score(T, p, average=k, zero_division=0) * 100, 2) for k in ('macro', 'weighted', 'micro', 'samples')}
res = dict(run=RUN_NAME, model=MODEL_NAME, best_epoch=int(max(history, key=lambda x: x['val_f1'])['epoch']),
           val=scores(T_val, P_val), test=scores(T_test, P_test))

# So với ViSoBERT ekman7 của Khoa (cùng tập, cùng thứ tự câu) và trung bình xác suất 2 mô hình
vb = os.path.join(VISOBERT_RUN, 'test_probs.npy')
if os.path.exists(vb) and not SMOKE_TEST:
    Pv_val, Pv_test = np.load(os.path.join(VISOBERT_RUN, 'val_probs.npy')), np.load(vb)
    assert (np.load(os.path.join(VISOBERT_RUN, 'test_labels.npy')) == T_test).all(), 'Thứ tự câu khác run ViSoBERT'
    res['visobert'] = dict(val=scores(T_val, Pv_val), test=scores(T_test, Pv_test))
    res['trung_binh_voi_visobert'] = dict(val=scores(T_val, (P_val + Pv_val) / 2), test=scores(T_test, (P_test + Pv_test) / 2))
json.dump(res, open(os.path.join(OUT_DIR, 'ket_qua.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(json.dumps(res, ensure_ascii=False, indent=1))

## 12. Đóng gói để gửi (không kèm `best.pt` vì rất nặng; gửi riêng nếu cần)

In [ ]:
pack = os.path.join(OUT_ROOT, RUN_NAME + '_ketqua')
keep = [f for f in os.listdir(OUT_DIR) if not f.endswith('.pt') and not f.endswith('.tmp')]
os.makedirs(pack, exist_ok=True)
for f in keep:
    shutil.copy(os.path.join(OUT_DIR, f), os.path.join(pack, f))
zip_path = shutil.make_archive(pack, 'zip', pack); shutil.rmtree(pack)
print('File gửi kết quả:', zip_path)
print('Checkpoint (gửi riêng nếu cần):', BEST, f'({os.path.getsize(BEST)/1e9:.2f} GB)')